# GeoVision — ResNet18 classifier training (Kaggle)

Runs `ai/training/train_classifier.py`'s real training loop (Module 07) on a Kaggle GPU,
answering Open-Questions Q7 — the team has Kaggle access but had not run training there before.

**Before running, upload two Kaggle Datasets and attach them to this notebook** (the sidebar's
"Add Input" button):

1. **`geovision-ai-src`** — a zip of this repo's `ai/` folder (the `ai/src/`, `ai/pyproject.toml`,
   everything needed to `pip install -e` the package). Re-upload a new version whenever the code
   changes; Kaggle Datasets are versioned, so old runs stay reproducible against the version they
   used.
2. **`geovision-dataset-processed`** — a zip of `dataset/processed/` (the output of
   `scripts/split_dataset.py`, run locally first: `cd ai && uv run python
   ../scripts/split_dataset.py`).

**Also set the accelerator**: Notebook settings (right sidebar) → Accelerator → **GPU T4 x2** (or
whichever GPU is available on your account). Training will still run on CPU if you forget, just
much slower — Module 07's whole point is that both paths work.

Edit the two paths in the **Configuration** cell below to match your actual dataset slugs
(visible under "Add Input" once attached), then Run All.


In [ ]:
# --- Sanity check: confirm the GPU is actually visible before spending an hour training on CPU by accident ---
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU attached — check Settings > Accelerator, or proceed on CPU (slower, still correct).")


In [ ]:
# --- Install the ai package WITHOUT its pinned dependencies ---
# ai/pyproject.toml pins torch/torchvision to the CPU-only wheel index (ADR-012) — the API
# process must never need CUDA. Installing that here would silently REPLACE Kaggle's
# GPU-enabled torch with a CPU-only build. --no-deps keeps Kaggle's preinstalled torch/
# torchvision/scikit-learn exactly as they are; only the handful of packages the ai package
# actually needs that Kaggle doesn't already ship are installed explicitly below.
!cp -r /kaggle/input/geovision-ai-src/ai /kaggle/working/ai
!pip install -e /kaggle/working/ai --no-deps -q
!pip install -q albumentations


In [ ]:
# --- Configuration: edit these two paths to match your attached datasets ---
from pathlib import Path

PROCESSED_ROOT = Path("/kaggle/input/geovision-dataset-processed/processed")
RUN_DIR = Path("/kaggle/working/outputs/runs/resnet18-kaggle")

assert PROCESSED_ROOT.is_dir(), f"{PROCESSED_ROOT} not found — check the dataset slug in Add Input"
print(sorted(p.name for p in PROCESSED_ROOT.iterdir()))  # expect: test, train, validation


In [ ]:
# --- Train (Module-07-Classifier-Training.md's recipe, ai/training/trainer.py) ---
from ai.training.trainer import TrainingConfig, train

config = TrainingConfig(
    processed_root=PROCESSED_ROOT,
    run_dir=RUN_DIR,
    epochs=60,          # early stopping (patience=10) will usually end it sooner
    device="auto",      # resolves to cuda automatically when the accelerator is attached
    num_workers=2,       # Kaggle's Linux kernel doesn't need Windows' num_workers=0 caution
)
result = train(config)
print(result)


## Getting the checkpoint out

`RUN_DIR` (under `/kaggle/working/`) is exactly what Kaggle's **Output** tab shows after the
notebook finishes — `best.pt`, `last.pt`, `metrics.csv`, `confusion_matrix.json`, `config.json`
are all there to download directly from the UI, no extra step needed.

**Where it lives long-term (Open-Questions Q10):** publish `best.pt` as a **Kaggle Dataset or
Model** straight from this notebook's Output tab (zero extra upload step, since it is already
sitting in `/kaggle/working/`) *and* attach it to a **GitHub Release** for the version that
actually goes in front of the panel — the two are complementary, not a choice between them.


In [ ]:
# --- Quick look at how training went ---
import pandas as pd

metrics = pd.read_csv(RUN_DIR / "metrics.csv")
print(metrics.tail(10))
print(f"\nBest validation macro-F1: {result.best_macro_f1:.4f}")
